# Решения: Производная как скорость изменения

**Для преподавателя.** Ниже по разделам разобраны все задачи `lesson.ipynb` и `homework.ipynb`. Не выдавать до сдачи.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relief(x):
    return 0.08 * (x - 3) ** 3 + 0.6 * (x - 3) ** 2 + 2

xs = np.linspace(-2, 8, 201)
ys = relief(xs)


## Урок. 1. Средняя скорость на отрезке

In [ ]:
x_left, x_right = 0.0, 2.0
average_rate = (relief(x_right) - relief(x_left)) / (x_right - x_left)
assert np.isfinite(average_rate)
print(round(float(average_rate), 4))


## Урок. 2. Две оценки производной

In [ ]:
def derivative_forward(fun, x, h=1e-3):
    return float((fun(x + h) - fun(x)) / h)

def derivative_central(fun, x, h=1e-3):
    return float((fun(x + h) - fun(x - h)) / (2 * h))

d_forward = derivative_forward(relief, 1.0)
d_central = derivative_central(relief, 1.0)
assert abs(d_forward - d_central) < 0.02
print(round(d_forward, 6), round(d_central, 6))


## Урок. 3. Эксперимент с шагом h

In [ ]:
h_values = [1.0, 0.1, 0.01, 0.001]
estimates = [derivative_central(relief, 1.0, h) for h in h_values]
assert len(estimates) == len(h_values)
print(list(zip(h_values, [round(value, 6) for value in estimates])))


## Урок. 4. Карта направлений

In [ ]:
probe_points = [-1.0, 0.0, 2.0, 4.0, 6.0]
directions = []
for point in probe_points:
    slope = derivative_central(relief, point)
    directions.append("up" if slope > 0.05 else "down" if slope < -0.05 else "flat")
assert len(directions) == len(probe_points)
print(list(zip(probe_points, directions)))


## Урок. 5. График рельефа и точек

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, ys, label="relief(x)")
ax.scatter(probe_points, [relief(x) for x in probe_points], color="crimson", label="probes")
ax.set(xlabel="x", ylabel="height", title="Рельеф виртуального полигона")
ax.legend()
assert len(ax.lines) >= 1 and ax.get_xlabel() and ax.get_ylabel()
plt.show()


## Урок. 6. Поиск почти горизонтальной точки

In [ ]:
scan = np.linspace(0.0, 7.0, 701)
scan_slopes = np.array([derivative_central(relief, x) for x in scan])
flat_index = int(np.argmin(np.abs(scan_slopes)))
flat_x = float(scan[flat_index])
flat_slope = float(scan_slopes[flat_index])
assert abs(flat_slope) < 0.02
print(flat_x, flat_slope)


## Урок. 7. Минимум или максимум

In [ ]:
left_slope = derivative_central(relief, flat_x - 0.1)
right_slope = derivative_central(relief, flat_x + 0.1)
if left_slope < 0 < right_slope:
    stationary_kind = "minimum"
elif left_slope > 0 > right_slope:
    stationary_kind = "maximum"
else:
    stationary_kind = "neither"
assert stationary_kind in {"minimum", "maximum", "neither"}
print(round(left_slope, 4), round(right_slope, 4), stationary_kind)


## Урок. 8. Открытый эксперимент: когда h слишком мал

In [ ]:
tiny_h_values = [1e-3, 1e-6, 1e-9, 1e-12]
tiny_estimates = [derivative_central(relief, 1.0, h) for h in tiny_h_values]
H_NOTE = (
    "Уменьшение h сначала почти не меняет оценку. При экстремально малом h вычитаются "
    "почти равные числа, поэтому ошибка округления становится заметной. Малый шаг полезен, "
    "но правило «чем меньше, тем лучше» для численных вычислений неверно."
)
assert len(H_NOTE) >= 120
print(tiny_estimates)
print(H_NOTE)


## Урок. 9. Самостоятельно: функция slope_map

In [ ]:
def slope_map(fun, points, h=1e-3, tolerance=0.05):
    result = []
    for point in points:
        slope = derivative_central(fun, point, h)
        direction = "up" if slope > tolerance else "down" if slope < -tolerance else "flat"
        result.append((point, direction))
    return result

mapped = slope_map(relief, [-1.0, 3.0, 5.0])
assert len(mapped) == 3
print(mapped)


## ДЗ. Данные и функции

In [ ]:
import numpy as np

def terrain(x):
    return 0.5 * (x + 1) ** 2 + np.sin(1.5 * x)

def derivative_central(fun, x, h=1e-3):
    return float((fun(x + h) - fun(x - h)) / (2 * h))


## ДЗ. Закрепление: скорости на новой функции

In [ ]:
average_rate = (terrain(3.0) - terrain(1.0)) / 2.0
slopes = [derivative_central(terrain, point) for point in (-2.0, 0.0, 2.0)]
assert len(slopes) == 3
print(round(average_rate, 4), [round(value, 4) for value in slopes])


## ДЗ. База: стационарные точки

In [ ]:
grid = np.linspace(-3.0, 3.0, 1201)
candidate_x = [float(x) for x in grid if abs(derivative_central(terrain, x)) < 0.02]
assert candidate_x
print(candidate_x[:10])


## ДЗ. Углубление: устойчивость к h

In [ ]:
test_h = [0.5, 0.1, 0.01, 0.001]
flat_by_h = []
for h in test_h:
    values = [abs(derivative_central(terrain, x, h)) for x in grid]
    flat_by_h.append(float(grid[int(np.argmin(values))]))
assert len(flat_by_h) == len(test_h)
print(list(zip(test_h, flat_by_h)))


## ДЗ. Вызов: несколько стационарных точек

In [ ]:
STATIONARY_NOTE = (
    "Сканирование дает группы соседних кандидатов, а не уникальные точные корни. Для каждого "
    "кластера я беру одну точку и сравниваю знак производной слева и справа. Переход «минус → плюс» "
    "означает локальный минимум, «плюс → минус» — локальный максимум. Такой тест различает тип "
    "стационарной точки, но его результат зависит от сетки, допуска и шага h."
)
READY = True
assert len(STATIONARY_NOTE) >= 220 and READY
print(STATIONARY_NOTE)
